In [ ]:
import os
import copy
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GATv2Conv
from torch_geometric.data import Data

warnings.filterwarnings("ignore")

# ==============================
# Path configuration
# Set environment variables to override these defaults.
# ==============================
PROJECT_ROOT = os.environ.get("DEEPGLOC_PROJECT_ROOT", os.getcwd())
AUGMENTED_GRAPH_DIR = os.environ.get(
    "DEEPGLOC_AUGMENTED_GRAPH_DIR",
    os.path.join(PROJECT_ROOT, "augmented_graph_output")
)
INPUT_PYG_FILE = os.environ.get(
    "DEEPGLOC_INPUT_PYG_FILE",
    os.path.join(AUGMENTED_GRAPH_DIR, "pyg_data_with_topology_features.pt")
)

OUT_DIR = os.environ.get(
    "DEEPGLOC_FINAL_EXPANSION_OUT_DIR",
    os.path.join(PROJECT_ROOT, "final_expansion_results")
)
MODEL_DIR = os.path.join(OUT_DIR, "models")
HISTORY_DIR = os.path.join(OUT_DIR, "history")
PRED_DIR = os.path.join(OUT_DIR, "predictions")
GENESET_DIR = os.path.join(OUT_DIR, "expanded_gene_sets")

for d in [OUT_DIR, MODEL_DIR, HISTORY_DIR, PRED_DIR, GENESET_DIR]:
    os.makedirs(d, exist_ok=True)

# ==============================
# 训练配置
# ==============================
CONFIG = {
    "hidden_dim": 32,
    "heads_1": 4,
    "heads_2": 1,
    "dropout": 0.25,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "num_epochs": 400,
    "patience": 50,
    "lambda_orth": 1e-3,
    "use_seed_weight": True,
    "label_smoothing": 0.0
}

# 多次重复训练，用于稳定性筛选
N_RUNS = 20
RUN_SEEDS = list(range(2026, 2026 + N_RUNS))

# 高可信扩展基因筛选阈值
CONF_THRES = {
    "min_mean_max_prob": 0.70,   # 平均最大类别概率
    "min_mean_margin": 0.15,     # 平均第一名-第二名概率差
    "min_pred_freq": 0.80        # 多次训练中落到同一类的频率
}

In [ ]:
def set_seed(seed: int = 2026):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(2026)

bundle = torch.load(INPUT_PYG_FILE, map_location="cpu", weights_only=False)

data = bundle["data"]
node_df = bundle["node_df"].copy()
edge_df = bundle["edge_df"].copy()
hard_seed_df = bundle.get("hard_seed_df", pd.DataFrame()).copy()
ambiguous_seed_df = bundle.get("ambiguous_seed_df", pd.DataFrame()).copy()
core_all = bundle.get("core_all", pd.DataFrame()).copy()
aux_info = bundle.get("aux_info", {})

print(data)
print("node_df shape:", node_df.shape)
print("edge_df shape:", edge_df.shape)
print("hard_seed_df shape:", hard_seed_df.shape)
print("ambiguous_seed_df shape:", ambiguous_seed_df.shape)
print("core_all shape:", core_all.shape)

In [ ]:
# process_map
if "process_map" in aux_info:
    process_map = aux_info["process_map"]
else:
    process_map = {
        "Biosynthesis": 0,
        "Esterification": 1,
        "Excretion": 2,
        "Uptake": 3
    }

label2process = {v: k for k, v in process_map.items()}
process_names = [label2process[i] for i in sorted(label2process)]

# 节点映射
if "idx2gene" in aux_info:
    idx2gene = aux_info["idx2gene"]
else:
    idx2gene = {i: str(g) for i, g in enumerate(node_df["ENSGID"].tolist())}

if "gene2idx" in aux_info:
    gene2idx = aux_info["gene2idx"]
else:
    gene2idx = {g: i for i, g in idx2gene.items()}

# 监督 mask：最终扩展阶段不再拆 train/val/test
if hasattr(data, "supervised_mask"):
    supervised_mask = data.supervised_mask.clone()
elif hasattr(data, "train_mask"):
    supervised_mask = data.train_mask.clone()
else:
    supervised_mask = (data.y >= 0)

data.supervised_mask = supervised_mask

# 全部 core mask
if hasattr(data, "all_core_mask"):
    all_core_mask = data.all_core_mask.clone()
else:
    all_core_mask = (data.y >= 0)

data.all_core_mask = all_core_mask

# seed weight
if not hasattr(data, "seed_weight"):
    data.seed_weight = torch.ones(data.num_nodes, dtype=torch.float32)

# edge_attr
if not hasattr(data, "edge_attr") or data.edge_attr is None:
    # 如果没有 edge_attr，至少给一个 edge_weight
    if hasattr(data, "edge_weight"):
        data.edge_attr = data.edge_weight.view(-1, 1).float()
    else:
        data.edge_attr = torch.ones(data.edge_index.shape[1], 1, dtype=torch.float32)

# 确保 node_df 有 ENSGID
if "ENSGID" not in node_df.columns:
    node_df["ENSGID"] = [idx2gene[i] for i in range(len(node_df))]

print("process_map:", process_map)
print("supervised seeds:", int(data.supervised_mask.sum()))
print("all core genes in graph:", int(data.all_core_mask.sum()))

for lab in sorted(label2process):
    n_lab = int(((data.y == lab) & data.supervised_mask).sum())
    print(label2process[lab], n_lab)

In [ ]:
class ExpansionGAT(nn.Module):
    def __init__(
        self,
        in_dim,
        edge_dim,
        hidden_dim,
        out_dim=4,
        heads_1=4,
        heads_2=1,
        dropout=0.25
    ):
        super().__init__()
        self.dropout = dropout

        self.gat1 = GATv2Conv(
            in_channels=in_dim,
            out_channels=hidden_dim,
            heads=heads_1,
            concat=True,
            dropout=dropout,
            edge_dim=edge_dim
        )

        self.gat2 = GATv2Conv(
            in_channels=hidden_dim * heads_1,
            out_channels=out_dim,
            heads=heads_2,
            concat=False,
            dropout=dropout,
            edge_dim=edge_dim
        )

    def forward(self, x, edge_index, edge_attr):
        x = self.gat1(x, edge_index, edge_attr)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        logits = self.gat2(x, edge_index, edge_attr)
        return logits

In [ ]:
def orthogonality_loss(logits):
    """
    logits: [N, C]
    鼓励4类输出不要高度塌缩到一起
    """
    probs = F.softmax(logits, dim=1)  # [N, C]
    gram = probs.T @ probs            # [C, C]
    eye = torch.eye(gram.size(0), device=gram.device)
    return torch.norm(gram - eye, p="fro")


def evaluate_supervised(model, data):
    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index, data.edge_attr)
        pred = logits.argmax(dim=1)

        mask = data.supervised_mask
        if int(mask.sum()) == 0:
            return {"loss": np.nan, "acc": np.nan}

        loss = F.cross_entropy(
            logits[mask],
            data.y[mask],
            label_smoothing=CONFIG["label_smoothing"]
        ).item()

        acc = (pred[mask] == data.y[mask]).float().mean().item()

    return {"loss": loss, "acc": acc}


def train_one_run(data_cpu, config, run_seed, run_name="run"):
    set_seed(run_seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data = copy.deepcopy(data_cpu).to(device)

    model = ExpansionGAT(
        in_dim=data.x.shape[1],
        edge_dim=data.edge_attr.shape[1],
        hidden_dim=config["hidden_dim"],
        out_dim=len(process_map),
        heads_1=config["heads_1"],
        heads_2=config["heads_2"],
        dropout=config["dropout"]
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_state = None
    best_monitor = np.inf
    best_epoch = -1
    wait = 0
    history = []

    for epoch in range(1, config["num_epochs"] + 1):
        model.train()
        optimizer.zero_grad()

        logits = model(data.x, data.edge_index, data.edge_attr)

        mask = data.supervised_mask

        ce_vec = F.cross_entropy(
            logits[mask],
            data.y[mask],
            reduction="none",
            label_smoothing=config["label_smoothing"]
        )

        if config["use_seed_weight"]:
            ce = (ce_vec * data.seed_weight[mask]).mean()
        else:
            ce = ce_vec.mean()

        orth = orthogonality_loss(logits)
        loss = ce + config["lambda_orth"] * orth

        loss.backward()
        optimizer.step()

        # 不再看 val_loss，而是看训练监督损失 total loss
        monitor = loss.item()

        eval_res = evaluate_supervised(model, data)

        row = {
            "epoch": epoch,
            "supervised_loss": eval_res["loss"],
            "supervised_acc": eval_res["acc"],
            "ce_loss": ce.item(),
            "orth_loss": orth.item(),
            "total_loss": loss.item()
        }
        history.append(row)

        if monitor < best_monitor:
            best_monitor = monitor
            best_epoch = epoch
            wait = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1

        if epoch % 20 == 0 or epoch == 1:
            print(
                f"[{run_name}] Epoch {epoch:03d} | "
                f"sup_loss={eval_res['loss']:.4f} sup_acc={eval_res['acc']:.4f} | "
                f"ce={ce.item():.4f} orth={orth.item():.4f} total={loss.item():.4f}"
            )

        if wait >= config["patience"]:
            print(f"[{run_name}] Early stopping at epoch {epoch}, best epoch = {best_epoch}")
            break

    model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index, data.edge_attr)
        prob = F.softmax(logits, dim=1)
        pred = prob.argmax(dim=1)

    history_df = pd.DataFrame(history)

    out_model_file = os.path.join(MODEL_DIR, f"{run_name}_best_model.pt")
    torch.save(model.state_dict(), out_model_file)

    out_hist_file = os.path.join(HISTORY_DIR, f"{run_name}_history.csv")
    history_df.to_csv(out_hist_file, index=False)

    return {
        "run_name": run_name,
        "seed": run_seed,
        "best_epoch": best_epoch,
        "best_monitor": best_monitor,
        "history_df": history_df,
        "supervised_final": evaluate_supervised(model, data),
        "logits": logits.detach().cpu(),
        "prob": prob.detach().cpu(),
        "pred": pred.detach().cpu(),
        "model_path": out_model_file,
        "history_path": out_hist_file
    }

In [ ]:
all_results = []

for i, run_seed in enumerate(RUN_SEEDS, start=1):
    run_name = f"run_{i:02d}_seed_{run_seed}"
    print("\n" + "=" * 80)
    print("Training", run_name)
    print("=" * 80)

    res = train_one_run(
        data_cpu=data,
        config=CONFIG,
        run_seed=run_seed,
        run_name=run_name
    )
    all_results.append(res)

summary_rows = []
for res in all_results:
    row = {
        "run_name": res["run_name"],
        "seed": res["seed"],
        "best_epoch": res["best_epoch"],
        "best_monitor": res["best_monitor"],
        "supervised_loss": res["supervised_final"]["loss"],
        "supervised_acc": res["supervised_final"]["acc"]
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(OUT_DIR, "training_run_summary.csv"), index=False)

summary_df

In [ ]:
# 堆叠概率和预测
prob_stack = torch.stack([res["prob"] for res in all_results], dim=0)   # [R, N, C]
pred_stack = torch.stack([res["pred"] for res in all_results], dim=0)   # [R, N]

mean_prob = prob_stack.mean(dim=0)  # [N, C]
std_prob = prob_stack.std(dim=0)    # [N, C]

mean_pred = mean_prob.argmax(dim=1)  # [N]
max_prob, _ = mean_prob.max(dim=1)

# second prob
sorted_prob, _ = torch.sort(mean_prob, dim=1, descending=True)
second_prob = sorted_prob[:, 1]
margin = max_prob - second_prob

# 一致性频率：最终类别在多少次训练中都被预测到
pred_freq = []
for i in range(mean_prob.shape[0]):
    cls = int(mean_pred[i].item())
    freq = (pred_stack[:, i] == cls).float().mean().item()
    pred_freq.append(freq)
pred_freq = torch.tensor(pred_freq, dtype=torch.float32)

print("prob_stack shape:", prob_stack.shape)
print("mean_prob shape:", mean_prob.shape)

In [ ]:
pred_df = pd.DataFrame({
    "node_idx": np.arange(data.num_nodes),
    "ENSGID": [idx2gene[i] for i in range(data.num_nodes)],
    "pred_label": mean_pred.numpy(),
    "pred_process": [label2process[int(x)] for x in mean_pred.numpy()],
    "mean_max_prob": max_prob.numpy(),
    "mean_second_prob": second_prob.numpy(),
    "mean_margin": margin.numpy(),
    "pred_freq": pred_freq.numpy(),
    "is_supervised_seed": data.supervised_mask.cpu().numpy().astype(int),
    "is_all_core": data.all_core_mask.cpu().numpy().astype(int),
})

# 每类平均概率和标准差
for lab, proc in label2process.items():
    pred_df[f"prob_{proc}"] = mean_prob[:, lab].numpy()
    pred_df[f"prob_sd_{proc}"] = std_prob[:, lab].numpy()

# 非seed/unlabeled标记
pred_df["is_unlabeled_target"] = ((pred_df["is_all_core"] == 0)).astype(int)

pred_df.to_csv(os.path.join(PRED_DIR, "all_gene_predictions_meanprob.csv"), index=False)
pred_df.head()

In [ ]:
credible_mask = (
    (pred_df["is_unlabeled_target"] == 1) &
    (pred_df["mean_max_prob"] >= CONF_THRES["min_mean_max_prob"]) &
    (pred_df["mean_margin"] >= CONF_THRES["min_mean_margin"]) &
    (pred_df["pred_freq"] >= CONF_THRES["min_pred_freq"])
)

credible_df = pred_df.loc[credible_mask].copy()

# 综合排序分数：可自定义
credible_df["confidence_score"] = (
    0.5 * credible_df["mean_max_prob"] +
    0.3 * credible_df["pred_freq"] +
    0.2 * credible_df["mean_margin"]
)

credible_df = credible_df.sort_values(
    ["pred_process", "confidence_score", "mean_max_prob"],
    ascending=[True, False, False]
).reset_index(drop=True)

credible_df.to_csv(os.path.join(PRED_DIR, "credible_expansion_candidates.csv"), index=False)

print("credible expansion genes:", credible_df.shape[0])
credible_df.head()

In [ ]:
expanded_sets = {}

for proc in process_names:
    df_proc = credible_df[credible_df["pred_process"] == proc].copy()
    df_proc = df_proc.sort_values(
        ["confidence_score", f"prob_{proc}", "pred_freq", "mean_margin"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    expanded_sets[proc] = df_proc

    out_file = os.path.join(GENESET_DIR, f"expanded_genes_{proc}.csv")
    df_proc.to_csv(out_file, index=False)

    print(f"{proc}: {df_proc.shape[0]} genes -> {out_file}")

In [ ]:
strict_mask = (
    (pred_df["is_unlabeled_target"] == 1) &
    (pred_df["mean_max_prob"] >= 0.80) &
    (pred_df["mean_margin"] >= 0.20) &
    (pred_df["pred_freq"] >= 0.90)
)

lenient_mask = (
    (pred_df["is_unlabeled_target"] == 1) &
    (pred_df["mean_max_prob"] >= 0.60) &
    (pred_df["mean_margin"] >= 0.10) &
    (pred_df["pred_freq"] >= 0.70)
)

strict_df = pred_df.loc[strict_mask].copy()
lenient_df = pred_df.loc[lenient_mask].copy()

strict_df.to_csv(os.path.join(PRED_DIR, "credible_expansion_candidates_strict.csv"), index=False)
lenient_df.to_csv(os.path.join(PRED_DIR, "credible_expansion_candidates_lenient.csv"), index=False)

print("strict genes:", strict_df.shape[0])
print("default genes:", credible_df.shape[0])
print("lenient genes:", lenient_df.shape[0])

In [ ]:
for proc in process_names:
    df_proc = expanded_sets[proc].copy()

    gene_list_file = os.path.join(GENESET_DIR, f"expanded_gene_list_{proc}.txt")
    with open(gene_list_file, "w", encoding="utf-8") as f:
        for g in df_proc["ENSGID"].tolist():
            f.write(str(g) + "\n")

    print(proc, "gene list saved:", gene_list_file)

In [ ]:
import itertools
import pandas as pd

# 先把每个过程的 ENSGID 提成 set
expanded_gene_dict = {}
for proc, df_proc in expanded_sets.items():
    genes = set(df_proc["ENSGID"].astype(str).tolist())
    expanded_gene_dict[proc] = genes

# 两两交集检查
overlap_rows = []
for p1, p2 in itertools.combinations(expanded_gene_dict.keys(), 2):
    g1 = expanded_gene_dict[p1]
    g2 = expanded_gene_dict[p2]

    inter = g1 & g2
    union = g1 | g2
    jaccard = len(inter) / len(union) if len(union) > 0 else 0.0

    overlap_rows.append({
        "set1": p1,
        "set2": p2,
        "n_set1": len(g1),
        "n_set2": len(g2),
        "n_intersection": len(inter),
        "jaccard": jaccard
    })

overlap_df = pd.DataFrame(overlap_rows).sort_values(
    ["n_intersection", "jaccard"], ascending=[False, False]
).reset_index(drop=True)

overlap_df

In [ ]:
pairwise_overlap_genes = {}

for p1, p2 in itertools.combinations(expanded_gene_dict.keys(), 2):
    inter = sorted(expanded_gene_dict[p1] & expanded_gene_dict[p2])
    pairwise_overlap_genes[(p1, p2)] = inter

for (p1, p2), genes in pairwise_overlap_genes.items():
    print(f"\n{p1} vs {p2} | overlap = {len(genes)}")
    if len(genes) > 0:
        print(genes[:20])  # 只先看前20个

In [ ]:
overlap_df.to_csv(os.path.join(GENESET_DIR, "pairwise_overlap_summary.csv"), index=False)

overlap_gene_rows = []
for (p1, p2), genes in pairwise_overlap_genes.items():
    for g in genes:
        overlap_gene_rows.append({
            "set1": p1,
            "set2": p2,
            "ENSGID": g
        })

overlap_gene_df = pd.DataFrame(overlap_gene_rows)
overlap_gene_df.to_csv(os.path.join(GENESET_DIR, "pairwise_overlap_genes.csv"), index=False)

print("saved:",
      os.path.join(GENESET_DIR, "pairwise_overlap_summary.csv"))
print("saved:",
      os.path.join(GENESET_DIR, "pairwise_overlap_genes.csv"))

In [ ]:
total_overlap = overlap_df["n_intersection"].sum()
print("Total pairwise overlap count:", total_overlap)

if total_overlap == 0:
    print("扩展基因集两两互斥，没有交集。")
else:
    print("发现扩展基因集之间有交集，请检查 ENSGID 标准化或筛选逻辑。")

In [ ]:
# core_all 里每个过程的核心基因
core_gene_dict = {}
for proc in process_names:
    core_gene_dict[proc] = set(
        core_all.loc[core_all["process"] == proc, "ENSGID"].astype(str).tolist()
    )

# 最终用于打分的基因集 = core + expanded
final_gene_dict = {}
for proc in process_names:
    final_gene_dict[proc] = core_gene_dict[proc] | expanded_gene_dict[proc]

final_overlap_rows = []
for p1, p2 in itertools.combinations(final_gene_dict.keys(), 2):
    g1 = final_gene_dict[p1]
    g2 = final_gene_dict[p2]
    inter = g1 & g2
    union = g1 | g2
    jaccard = len(inter) / len(union) if len(union) > 0 else 0.0

    final_overlap_rows.append({
        "set1": p1,
        "set2": p2,
        "n_set1": len(g1),
        "n_set2": len(g2),
        "n_intersection": len(inter),
        "jaccard": jaccard
    })

final_overlap_df = pd.DataFrame(final_overlap_rows).sort_values(
    ["n_intersection", "jaccard"], ascending=[False, False]
).reset_index(drop=True)

final_overlap_df

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, PowerNorm

custom_cmap = LinearSegmentedColormap.from_list(
    "blue_green_yellow_orange_red_soft",
    [
        "#DCEAF7",  # 浅蓝（低值）
        "#6AAED6",  # 蓝
        '#5ec078',  # 绿
        "#F6E58D",  # 黄
        "#F5A65B",  # 橙黄
       "#F26966"   # 红（高值）
    ],
    N=256
)

In [ ]:
def plot_heatmap_pdf(
    mat_df,
    title,
    out_file,
    cmap=custom_cmap,
    vmin=None,
    vmax=None,
    annot_fmt=None,
    figsize=(7, 6),
    cbar_label="Value"
):
    fig, ax = plt.subplots(figsize=figsize)

    im = ax.imshow(
        mat_df.values,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        aspect="auto"
    )

    nrow, ncol = mat_df.shape

    ax.set_xticks(np.arange(ncol))
    ax.set_yticks(np.arange(nrow))
    ax.set_xticklabels(mat_df.columns, rotation=45, ha="right")
    ax.set_yticklabels(mat_df.index)

    ax.set_title(title)

    # 写数值
    for i in range(nrow):
        for j in range(ncol):
            val = mat_df.iloc[i, j]
            if annot_fmt is None:
                txt = str(val)
            else:
                txt = format(val, annot_fmt)

            ax.text(j, i, txt, ha="center", va="center", color="black")

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)

    plt.tight_layout()
    plt.savefig(out_file, format="pdf", bbox_inches="tight")
    plt.show()

    print("saved:", out_file)

In [ ]:
expanded_gene_dict = {}
for proc, df_proc in expanded_sets.items():
    expanded_gene_dict[proc] = set(df_proc["ENSGID"].astype(str).tolist())

process_order = ["Biosynthesis","Uptake", "Excretion", "Esterification"]
n = len(process_order)

intersection_mat = pd.DataFrame(
    np.zeros((n, n), dtype=int),
    index=process_order,
    columns=process_order
)

jaccard_mat = pd.DataFrame(
    np.zeros((n, n), dtype=float),
    index=process_order,
    columns=process_order
)

for p1 in process_order:
    for p2 in process_order:
        g1 = expanded_gene_dict[p1]
        g2 = expanded_gene_dict[p2]

        inter = g1 & g2
        union = g1 | g2

        intersection_mat.loc[p1, p2] = len(inter)
        jaccard_mat.loc[p1, p2] = len(inter) / len(union) if len(union) > 0 else 0.0

print("Intersection matrix:")
display(intersection_mat)

print("Jaccard matrix:")
display(jaccard_mat)

In [ ]:
out_file1 = os.path.join(GENESET_DIR, "expanded_sets_overlap_count_heatmap.pdf")

plot_heatmap_pdf(
    mat_df=intersection_mat,
    title="Overlap Count Heatmap of Expanded Gene Sets",
    out_file=out_file1,
    cmap=custom_cmap,
    vmin=intersection_mat.values.min(),
    vmax=intersection_mat.values.max(),
    annot_fmt="d",
    figsize=(7, 6),
    cbar_label="Intersection Count"
)

In [ ]:
out_file2 = os.path.join(GENESET_DIR, "expanded_sets_jaccard_heatmap.pdf")

plot_heatmap_pdf(
    mat_df=jaccard_mat,
    title="Jaccard Heatmap of Expanded Gene Sets",
    out_file=out_file2,
    cmap=custom_cmap,
    vmin=0,
    vmax=1,
    annot_fmt=".2f",
    figsize=(7, 6),
    cbar_label="Jaccard Index"
)

In [ ]:
core_gene_dict = {}
for proc in process_order:
    core_gene_dict[proc] = set(
        core_all.loc[core_all["process"] == proc, "ENSGID"].astype(str).tolist()
    )

final_gene_dict = {}
for proc in process_order:
    final_gene_dict[proc] = core_gene_dict[proc] | expanded_gene_dict[proc]

final_intersection_mat = pd.DataFrame(
    np.zeros((n, n), dtype=int),
    index=process_order,
    columns=process_order
)

final_jaccard_mat = pd.DataFrame(
    np.zeros((n, n), dtype=float),
    index=process_order,
    columns=process_order
)

for p1 in process_order:
    for p2 in process_order:
        g1 = final_gene_dict[p1]
        g2 = final_gene_dict[p2]

        inter = g1 & g2
        union = g1 | g2

        final_intersection_mat.loc[p1, p2] = len(inter)
        final_jaccard_mat.loc[p1, p2] = len(inter) / len(union) if len(union) > 0 else 0.0

print("Final intersection matrix (core + expanded):")
display(final_intersection_mat)

print("Final Jaccard matrix (core + expanded):")
display(final_jaccard_mat)

In [ ]:
out_file3 = os.path.join(GENESET_DIR, "final_sets_overlap_count_heatmap.pdf")

plot_heatmap_pdf(
    mat_df=final_intersection_mat,
    title="Overlap Count Heatmap of Final Gene Sets (Core + Expanded)",
    out_file=out_file3,
    cmap=custom_cmap,
    vmin=final_intersection_mat.values.min(),
    vmax=final_intersection_mat.values.max(),
    annot_fmt="d",
    figsize=(7, 6),
    cbar_label="Intersection Count"
)

In [ ]:
out_file4 = os.path.join(GENESET_DIR, "final_sets_jaccard_heatmap.pdf")

plot_heatmap_pdf(
    mat_df=final_jaccard_mat,
    title="Jaccard Heatmap of Final Gene Sets (Core + Expanded)",
    out_file=out_file4,
    cmap=custom_cmap,
    vmin=0,
    vmax=1,
    annot_fmt=".2f",
    figsize=(7, 6),
    cbar_label="Jaccard Index"
)

In [ ]:
import os
import numpy as np
import pandas as pd

# 固定过程顺序
process_order = ["Biosynthesis", "Uptake", "Excretion", "Esterification"]

# 输出目录
FINAL_SET_DIR = os.path.join(OUT_DIR, "final_gene_sets_with_symbol")
os.makedirs(FINAL_SET_DIR, exist_ok=True)

# -------------------------------------------------
# 1. 构建 ENSGID -> SYMBOL 映射
# -------------------------------------------------
symbol_map = {}

# 1) 先用 core_all 里的 SYMBOL
if "SYMBOL" in core_all.columns:
    tmp = core_all[["ENSGID", "SYMBOL"]].dropna().copy()
    tmp["ENSGID"] = tmp["ENSGID"].astype(str)
    tmp["SYMBOL"] = tmp["SYMBOL"].astype(str)
    symbol_map.update(dict(zip(tmp["ENSGID"], tmp["SYMBOL"])))

# 2) 再尝试从 node_df 里找 SYMBOL 列
candidate_symbol_cols = ["SYMBOL", "symbol", "GeneSymbol", "gene_symbol", "gene_name"]
node_symbol_col = None
for c in candidate_symbol_cols:
    if c in node_df.columns:
        node_symbol_col = c
        break

if node_symbol_col is not None and "ENSGID" in node_df.columns:
    tmp = node_df[["ENSGID", node_symbol_col]].dropna().copy()
    tmp["ENSGID"] = tmp["ENSGID"].astype(str)
    tmp[node_symbol_col] = tmp[node_symbol_col].astype(str)
    for g, s in zip(tmp["ENSGID"], tmp[node_symbol_col]):
        if g not in symbol_map or symbol_map[g] in ["", "nan", "None"]:
            symbol_map[g] = s

print("symbol_map size:", len(symbol_map))
print("node_df symbol column used:", node_symbol_col)

In [ ]:
def build_process_gene_sets(expanded_df, set_name, core_all, symbol_map, out_dir, process_order):
    """
    expanded_df: strict_df / credible_df(default) / lenient_df
    set_name: 'strict' / 'default' / 'lenient'
    """
    this_dir = os.path.join(out_dir, set_name)
    os.makedirs(this_dir, exist_ok=True)

    summary_rows = []

    for proc in process_order:
        # -----------------------------
        # A. core genes of this process
        # -----------------------------
        core_proc = core_all.loc[core_all["process"] == proc].copy()

        core_proc["ENSGID"] = core_proc["ENSGID"].astype(str)
        if "SYMBOL" not in core_proc.columns:
            core_proc["SYMBOL"] = core_proc["ENSGID"].map(symbol_map).fillna(core_proc["ENSGID"])
        else:
            core_proc["SYMBOL"] = core_proc["SYMBOL"].fillna(core_proc["ENSGID"])
            core_proc["SYMBOL"] = core_proc["SYMBOL"].astype(str)

        core_proc = core_proc[["ENSGID", "SYMBOL"]].drop_duplicates().copy()
        core_proc["process"] = proc
        core_proc["gene_source"] = "core"
        core_proc["set_level"] = set_name

        # -----------------------------
        # B. expanded genes of this process
        # -----------------------------
        exp_proc = expanded_df.loc[expanded_df["pred_process"] == proc].copy()

        if exp_proc.shape[0] > 0:
            exp_proc["ENSGID"] = exp_proc["ENSGID"].astype(str)
            exp_proc["SYMBOL"] = exp_proc["ENSGID"].map(symbol_map).fillna(exp_proc["ENSGID"])

            keep_cols = [
                "ENSGID", "SYMBOL", "pred_process",
                "mean_max_prob", "mean_second_prob", "mean_margin", "pred_freq"
            ]
            keep_cols = [c for c in keep_cols if c in exp_proc.columns]
            exp_proc = exp_proc[keep_cols].drop_duplicates(subset=["ENSGID"]).copy()

            exp_proc = exp_proc.rename(columns={"pred_process": "process"})
            exp_proc["gene_source"] = "expanded"
            exp_proc["set_level"] = set_name
        else:
            exp_proc = pd.DataFrame(columns=[
                "ENSGID", "SYMBOL", "process",
                "mean_max_prob", "mean_second_prob", "mean_margin", "pred_freq",
                "gene_source", "set_level"
            ])

        # -----------------------------
        # C. final genes = core + expanded
        # -----------------------------
        final_core = core_proc.copy()
        final_core["mean_max_prob"] = np.nan
        final_core["mean_second_prob"] = np.nan
        final_core["mean_margin"] = np.nan
        final_core["pred_freq"] = np.nan

        final_proc = pd.concat([final_core, exp_proc], axis=0, ignore_index=True)
        final_proc = final_proc.drop_duplicates(subset=["ENSGID"], keep="first").copy()

        # 如果你想让 expanded 排前面，可以自己改排序规则
        final_proc["source_rank"] = final_proc["gene_source"].map({"core": 0, "expanded": 1}).fillna(9)
        final_proc = final_proc.sort_values(
            ["source_rank", "ENSGID"],
            ascending=[True, True]
        ).drop(columns=["source_rank"]).reset_index(drop=True)

        # -----------------------------
        # D. 保存文件
        # -----------------------------
        core_file = os.path.join(this_dir, f"{proc}_core_genes_{set_name}.csv")
        exp_file = os.path.join(this_dir, f"{proc}_expanded_genes_{set_name}.csv")
        final_file = os.path.join(this_dir, f"{proc}_final_genes_core_plus_expanded_{set_name}.csv")

        core_proc.to_csv(core_file, index=False)
        exp_proc.to_csv(exp_file, index=False)
        final_proc.to_csv(final_file, index=False)

        # 也顺便导出 txt 列表
        with open(os.path.join(this_dir, f"{proc}_final_gene_list_{set_name}.txt"), "w", encoding="utf-8") as f:
            for g in final_proc["ENSGID"].tolist():
                f.write(str(g) + "\n")

        # -----------------------------
        # E. 汇总统计
        # -----------------------------
        summary_rows.append({
            "set_level": set_name,
            "process": proc,
            "n_core_genes": core_proc["ENSGID"].nunique(),
            "n_expanded_genes": exp_proc["ENSGID"].nunique(),
            "n_final_genes": final_proc["ENSGID"].nunique(),
            "core_file": core_file,
            "expanded_file": exp_file,
            "final_file": final_file
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_file = os.path.join(this_dir, f"summary_{set_name}.csv")
    summary_df.to_csv(summary_file, index=False)

    print(f"[{set_name}] saved summary ->", summary_file)
    return summary_df

In [ ]:
strict_summary_df = build_process_gene_sets(
    expanded_df=strict_df,
    set_name="strict",
    core_all=core_all,
    symbol_map=symbol_map,
    out_dir=FINAL_SET_DIR,
    process_order=process_order
)

default_summary_df = build_process_gene_sets(
    expanded_df=credible_df,
    set_name="default",
    core_all=core_all,
    symbol_map=symbol_map,
    out_dir=FINAL_SET_DIR,
    process_order=process_order
)

lenient_summary_df = build_process_gene_sets(
    expanded_df=lenient_df,
    set_name="lenient",
    core_all=core_all,
    symbol_map=symbol_map,
    out_dir=FINAL_SET_DIR,
    process_order=process_order
)

display(strict_summary_df)
display(default_summary_df)
display(lenient_summary_df)

In [ ]:
all_summary_df = pd.concat(
    [strict_summary_df, default_summary_df, lenient_summary_df],
    axis=0,
    ignore_index=True
)

all_summary_file = os.path.join(FINAL_SET_DIR, "all_set_levels_summary.csv")
all_summary_df.to_csv(all_summary_file, index=False)

print("saved:", all_summary_file)
all_summary_df

In [ ]:
print(node_symbol_col)